In [1]:
# 1. Kutubxonalar va ma'lumot
import time
import pandas as pd
import soundfile as sf
import torch
import jiwer
from transformers import WhisperProcessor, WhisperForConditionalGeneration

import sys, importlib
sys.path.insert(0, "../04_normalize")   # normalize.py shu papkada
import normalize
importlib.reload(normalize)

df = pd.read_csv("../04_normalize/manifest_tayyor.csv")
print("Audiolar:", len(df))

<python>\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Audiolar: 20737


In [2]:
# 2. Whisper tokenizer: token uzunligini tekshirish
MODEL = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(MODEL)
tokenizer = processor.tokenizer

df["tokenlar"] = [len(tokenizer(t).input_ids) for t in df["matn_model"]]
df["sozlar"] = df["matn_model"].str.split().str.len()

print(df["tokenlar"].describe().round(1))
print("448 dan oshgan:", (df["tokenlar"] > 448).sum())
print("Bir so'zga o'rtacha token:", round(df["tokenlar"].sum() / df["sozlar"].sum(), 2))

# BPE qoraqalpoq so'zini qanday bo'laklaydi
for soz in ["qaraqalpaqstan", "mámleketlik", "jaslarǵa", "hám"]:
    print(soz, "->", tokenizer.tokenize(soz))

<python>\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in ~\.cache\huggingface\hub\models--openai--whisper-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


count    20737.0
mean       118.0
std         32.3
min         53.0
25%         94.0
50%        114.0
75%        138.0
max        252.0
Name: tokenlar, dtype: float64
448 dan oshgan: 0
Bir so'zga o'rtacha token: 3.64
qaraqalpaqstan -> ['q', 'ara', 'q', 'al', 'pa', 'q', 'stan']
mámleketlik -> ['m', 'Ã¡m', 'le', 'ket', 'lik']
jaslarǵa -> ['jas', 'lar', 'Ç', 'µ', 'a']
hám -> ['h', 'Ã¡m']


In [3]:
# 3. Baholash uchun test'dan 50 ta klip
eval50 = df[df["split"] == "test"].sample(50, random_state=42).reset_index(drop=True)
eval50[["split", "idx", "fayl", "yangi_s", "matn", "matn_model"]].to_csv("eval_50.csv", index=False)
print("Jami davomiylik:", round(eval50["yangi_s"].sum() / 60, 1), "daqiqa")

Jami davomiylik: 13.7 daqiqa


In [4]:
# 4. Modelni yuklash (birinchi marta ~1 GB yuklab olinadi)
model = WhisperForConditionalGeneration.from_pretrained(MODEL)
model.eval()


def transkripsiya(fayl, til):
    wav, sr = sf.read("../" + fayl, dtype="float32")   # fayl yo'li loyiha ildiziga nisbatan
    kirish = processor(wav, sampling_rate=sr, return_tensors="pt").input_features
    with torch.no_grad():
        chiqish = model.generate(kirish, language=til, task="transcribe", max_new_tokens=440)
    return processor.batch_decode(chiqish, skip_special_tokens=True)[0]


# Bitta klipda sinov
print(transkripsiya(eval50.loc[0, "fayl"], "uz"))

You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


 Mwen legey edmuz, başlisun islam o bot kalasna da ajaratilgarezdiyen tisyaasda da Pakistan Islam disbublu yukas baslam enstere n'o rum bas hara, sqasle minsterga ishok dhor kutubol da.


In [5]:
# 5. 50 ta klipda ikki til tokeni bilan (CPU'da ~15-20 daqiqa)
for til in ["uz", "kk"]:
    boshlanish = time.time()
    natijalar = []
    for fayl in eval50["fayl"]:
        natijalar.append(transkripsiya(fayl, til))
    eval50[f"model_{til}"] = natijalar
    # Model matni ham bizning matn kabi normalizatsiya qilinadi, aks holda WER adolatsiz bo'ladi
    eval50[f"model_{til}_norm"] = [normalize.kichik_harf(t) for t in natijalar]
    print(til, "tayyor:", round((time.time() - boshlanish) / 60, 1), "daqiqa")

eval50.to_csv("eval_50_natija.csv", index=False)

uz tayyor: 6.1 daqiqa
kk tayyor: 9.2 daqiqa


In [6]:
# 6. WER va CER
haqiqiy = eval50["matn_model"].tolist()

for til in ["uz", "kk"]:
    taxmin = eval50[f"model_{til}_norm"].tolist()
    wer = jiwer.wer(haqiqiy, taxmin)
    cer = jiwer.cer(haqiqiy, taxmin)
    print(f"{til}:  WER = {wer:.1%}   CER = {cer:.1%}")

uz:  WER = 134.4%   CER = 58.5%
kk:  WER = 108.5%   CER = 99.1%


In [7]:
# 7. Misollar: model nima deb yozdi
for i in range(3):
    print("HAQIQIY:", eval50.loc[i, "matn_model"])
    print("UZ     :", eval50.loc[i, "model_uz_norm"])
    print("KK     :", eval50.loc[i, "model_kk_norm"])
    print()

HAQIQIY: mámleketimiz basshısın islamabad qalasında ajıratılǵan rezidenciyada pakistan islam respublikası bas ministriniń orınbasarı sırtqı isler ministri ishoq dor kútip aldı
UZ     : mwen legey edmuz başlisun islam o bot kalasna da ajaratilgarezdiyen tisyaasda da pakistan islam disbublu yukas baslam enstere n o rum bas hara sqasle minsterga ishok dhor kutubol da
KK     : мәмлеге едіміз баштысын исламу бұт қаласында ажатылғар ездеін тесісті діда пақистан исламды спубликаст бастыменстерін орым басары сүткыслен мистеріңі ешок дор күті болды

HAQIQIY: búgingi kúnde tórt million gektar egislik maydanı bir pútin onnan bes million tonna sıyımlılıqqa iye saqlaw qoymaxanaları hám úsh pútin onnan bes million tonna ónimdi qayta islew quwatlıqları xalıqtı arzan hám sapalı azıq awqat ónimleri menen támiyinlew sonday aq tórt milliard dollarlıq ónimdi eksport etiw imkaniyatın bermekte
UZ     : bukun deyek törd milion gektar yegislik maydanabriyarun milion donna isseimlalukka iyes hakla uttukima han